# LLM Endpoint Healthcheck

Bu notebook LLM tarafinda eksik olan seyin kutuphane mi, config mi, network/TLS mi, yoksa model response'u mu oldugunu ayirmak icin hazirlandi.

Key'i hucreye yazma. Ayarlar su sirayla okunur: terminal env, repo kokundeki `.env`, `llm/.env.local`, `secret/secrets.yaml`.

In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
raw_env_root = os.environ.get("EWS_ANOMALY_REPO_ROOT")
candidate_roots = []
if raw_env_root:
    candidate_roots.append(Path(raw_env_root).expanduser().resolve())
candidate_roots.extend([cwd, *cwd.parents])
candidate_roots.extend([(p / "ews-anomaly-detection").resolve() for p in [cwd, *cwd.parents]])

repo_root = next((p for p in dict.fromkeys(candidate_roots) if (p / "llm" / "llm_anomaly.py").exists()), None)
if repo_root is None:
    raise FileNotFoundError(f"Repo root bulunamadi. cwd={cwd}. Gerekirse EWS_ANOMALY_REPO_ROOT set et.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

default_secrets_path = repo_root / "secret" / "secrets.yaml"
os.environ.setdefault("EWS_ANOMALY_SECRETS_PATH", str(default_secrets_path))

print("initial_cwd=", cwd)
print("repo_root=", repo_root)
print("active_cwd=", Path.cwd().resolve())
print("EWS_ANOMALY_SECRETS_PATH=", os.environ.get("EWS_ANOMALY_SECRETS_PATH"))
print("secrets_exists=", Path(os.environ["EWS_ANOMALY_SECRETS_PATH"]).exists())

## 0. Secret dosyasi ve LLM section kontrolu

Bu hucre secret degerlerini basmaz. Sadece dosyanin okunup okunmadigini, `llm` bolumunu, secilen section'i ve key alaninin dolu olup olmadigini gosterir.

In [ ]:
from pathlib import Path
import os

import yaml

secrets_path = Path(os.environ["EWS_ANOMALY_SECRETS_PATH"]).expanduser().resolve()
print("secrets_path=", secrets_path)
print("secrets_exists=", secrets_path.exists())

if not secrets_path.exists():
    raise FileNotFoundError(f"Secret dosyasi yok: {secrets_path}")

with secrets_path.open("r", encoding="utf-8") as handle:
    secrets = yaml.safe_load(handle) or {}

print("secret_root_keys=", sorted(secrets.keys()))
llm_secret = secrets.get("llm") or {}
print("llm_section_exists=", isinstance(llm_secret, dict) and bool(llm_secret))
print("llm_root_keys=", sorted(llm_secret.keys()) if isinstance(llm_secret, dict) else [])

sections = llm_secret.get("sections") if isinstance(llm_secret, dict) else None
if isinstance(sections, dict):
    selected_section = os.environ.get("LLM_SECTION") or llm_secret.get("section") or llm_secret.get("default_section") or next(iter(sections), None)
    selected = sections.get(str(selected_section), {}) if selected_section is not None else {}
    print("llm_sections=", sorted(str(name) for name in sections.keys()))
    print("selected_llm_section=", selected_section)
else:
    selected_section = "llm_root"
    selected = llm_secret if isinstance(llm_secret, dict) else {}
    print("llm_sections=", [])
    print("selected_llm_section=", selected_section)

api_key = selected.get("api_key") or selected.get("openai_api_key")
print("selected_keys=", sorted(k for k in selected.keys() if "key" not in str(k).lower()))
print("api_key_present=", bool(api_key))
print("api_key_length=", len(str(api_key)) if api_key else 0)
print("base_url=", selected.get("base_url"))
print("model=", selected.get("model"))
print("timeout_seconds=", selected.get("timeout_seconds"))

## 1. Config okunuyor mu?

Bu hucre key'i basmaz. Sadece key'in yuklenip yuklenmedigini ve hangi endpoint/model kullanildigini gosterir.

In [ ]:
from urllib.parse import urlparse

from llm.llm_anomaly import build_llm_http_client, load_llm_settings, mask_url, proxy_env_present

settings = load_llm_settings()
parsed_base_url = urlparse(str(settings.get("base_url")))
safe_settings = {
    "base_url": mask_url(str(settings.get("base_url"))),
    "base_url_host": parsed_base_url.hostname,
    "model": settings.get("model"),
    "timeout_seconds": settings.get("timeout_seconds"),
    "client": "langchain_structured",
    "http_trust_env": settings.get("http_trust_env"),
    "proxy_env_present": proxy_env_present(),
    "ssl_verify": settings.get("ssl_verify"),
    "ca_bundle": settings.get("ca_bundle"),
    "settings_source": settings.get("source"),
    "api_key_loaded": bool(settings.get("api_key")),
}
if not safe_settings["api_key_loaded"]:
    raise RuntimeError("LLM API key okunamadi. Once ustteki '0. Secret dosyasi ve LLM section kontrolu' hucredeki secrets_path, llm_section_exists, selected_llm_section ve api_key_present alanlarina bak.")
safe_settings

## 2. TCP ve TLS handshake testi

Burada HTTP request atmadan once host'a baglanti ve TLS handshake denenir. Onceki hatadaki `_ssl.c:987 handshake operation timed out` bu adimda yakalanir.

In [ ]:
from urllib.parse import urlparse
import socket
import ssl
import time
import traceback

parsed = urlparse(str(settings["base_url"]))
host = parsed.hostname
port = parsed.port or (443 if parsed.scheme == "https" else 80)
connect_timeout = min(int(settings.get("timeout_seconds") or 120), 60)

print({"host": host, "port": port, "scheme": parsed.scheme, "connect_timeout": connect_timeout})
start = time.time()
try:
    raw_sock = socket.create_connection((host, port), timeout=connect_timeout)
    print("TCP OK", round(time.time() - start, 2), "sn")
    if parsed.scheme == "https":
        context = ssl.create_default_context()
        tls_start = time.time()
        with context.wrap_socket(raw_sock, server_hostname=host) as tls_sock:
            cert = tls_sock.getpeercert()
            print("TLS OK", round(time.time() - tls_start, 2), "sn", "version=", tls_sock.version())
            print("cert_subject=", cert.get("subject", [])[:1])
    else:
        raw_sock.close()
except Exception as exc:
    print("TCP/TLS FAILED:", repr(exc))
    traceback.print_exc(limit=2)

## 3. `/models` endpoint testi

`/models` 404/405 donebilir; bu tek basina chat'in calismadigi anlamina gelmez. Ama TLS timeout, 401 veya proxy hatalarini ayirmaya yarar.

In [ ]:
import json
import urllib.error
import urllib.request

models_url = str(settings["base_url"]).rstrip("/") + "/models"
request = urllib.request.Request(
    models_url,
    headers={"Authorization": f"Bearer {settings['api_key']}"},
    method="GET",
)

request_timeout = int(settings.get("timeout_seconds") or 300)
start = time.time()
try:
    with urllib.request.urlopen(request, timeout=request_timeout) as response:
        body = response.read().decode("utf-8", errors="replace")
    print("MODELS OK", {"status": response.status, "elapsed_sec": round(time.time() - start, 2)})
    print(body[:1000])
except urllib.error.HTTPError as exc:
    body = exc.read().decode("utf-8", errors="replace")
    print("MODELS HTTP ERROR", {"status": exc.code, "elapsed_sec": round(time.time() - start, 2)})
    print(body[:1000])
except Exception as exc:
    print("MODELS FAILED", {"elapsed_sec": round(time.time() - start, 2), "error": repr(exc)})
    traceback.print_exc(limit=2)

## 4. Repo'nun LangChain/Pydantic structured call testi

Bu test `run-oracle` icinde kullanilan ayni `ChatOpenAI + ChatPromptTemplate + with_structured_output(Pydantic)` zincirini cagirir. Basarili olursa structured karar JSON'u donmelidir.

In [ ]:
from llm.llm_anomaly import anomaly_batch_schema, build_langchain_structured_chain, invoke_langchain_structured_decision

healthcheck_evidence = {
    "mono_id": "HEALTHCHECK",
    "cohort_dt": "2026-05-31",
    "context": {"test_record": True},
    "decision_contract": {"target_or_label_available": False, "llm_should_decide_is_anomaly": True},
    "peer_definition": {"base": "healthcheck synthetic record"},
    "data_quality": {"coverage_ratio": 1.0, "customer_history_periods": 0, "caveat": "Synthetic endpoint healthcheck record."},
    "features": [
        {
            "name": "healthcheck_constant_signal",
            "dictionary": {"description": "Synthetic constant signal for endpoint healthcheck", "risk_direction": "NEUTRAL"},
            "current_value": 1.0,
            "history": {"period_count": 0, "median": None},
            "trend": {"trend_break_flag": False},
            "seasonality": {"available": False},
            "peer": {"peer_median": 1.0, "peer_z": 0.0, "peer_support": 1, "peer_quality": "TEST"},
            "data_quality": {"missing_flag": False},
        }
    ],
}

start = time.time()
try:
    schema = anomaly_batch_schema()
    print("STRUCTURED SCHEMA OK", schema.__name__)
    chain = build_langchain_structured_chain()
    result = invoke_langchain_structured_decision(chain, healthcheck_evidence)
    print("STRUCTURED CHAT OK", {"elapsed_sec": round(time.time() - start, 2)})
    print(json.dumps(result, ensure_ascii=False, indent=2))
except Exception as exc:
    print("STRUCTURED CHAT FAILED", {"elapsed_sec": round(time.time() - start, 2), "error": repr(exc)})
    traceback.print_exc(limit=2)

## 5. Opsiyonel ham LangChain testi

Normalde bu hucreyi calistirmana gerek yok. Repo akisi 4. hucrede zaten LangChain/Pydantic structured zinciriyle test edilir. Ham `ChatOpenAI.invoke` davranisini ayrica gormek istersen `RUN_RAW_LANGCHAIN_TEST = True` yap.

In [ ]:
RUN_RAW_LANGCHAIN_TEST = False

if not RUN_RAW_LANGCHAIN_TEST:
    print("RAW LANGCHAIN SKIPPED: repo structured chain testi yeterli. Gerekirse RUN_RAW_LANGCHAIN_TEST=True yap.")
else:
    try:
        from langchain_openai import ChatOpenAI

        llm_kwargs = {
            "base_url": str(settings["base_url"]).rstrip("/"),
            "api_key": settings["api_key"],
            "model": str(settings["model"]),
            "temperature": 0,
            "http_client": build_llm_http_client(settings),
        }
        lc_llm = ChatOpenAI(**llm_kwargs)
        start = time.time()
        response = lc_llm.invoke("Sadece JSON don: {\"ok\": true}")
        print("LANGCHAIN CHAT OK", {"elapsed_sec": round(time.time() - start, 2)})
        print(response.content)
    except ImportError as exc:
        print("LANGCHAIN SKIPPED: langchain_openai import edilemedi", repr(exc))
    except Exception as exc:
        print("LANGCHAIN CHAT FAILED", {"elapsed_sec": round(time.time() - start, 2), "error": repr(exc)})
        traceback.print_exc(limit=2)

## Sonuc nasil okunur?

- `api_key_loaded=False`: env veya `secret/secrets.yaml` LLM key okumuyor.
- `TCP/TLS FAILED` ve handshake timeout: kutuphane degil, endpoint/network/proxy/TLS/OpenShift route problemidir.
- `/models` 401: key/auth problemi.
- `/models` 404/405 ama `STRUCTURED CHAT OK`: problem yok, endpoint `/models` desteklemiyor olabilir.
- `STRUCTURED CHAT OK`: LLM endpoint repo akisi icin calisiyor.
- `STRUCTURED CHAT FAILED`: LangChain/Pydantic structured output zinciri hata verdi; hata body/trace'e bak.